# Gabarito — Módulo 4: Long Short-Term Memory (LSTM)

In [1]:
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

X_train_pad = np.load("X_train_pad.npy")
X_test_pad = np.load("X_test_pad.npy")
y_train = np.load("y_train.npy")
y_test = np.load("y_test.npy")
embedding_matrix = np.load("embedding_matrix.npy")
with open("config.json") as f:
    config = json.load(f)
test_text = pd.read_csv("test_text.csv")

print(X_train_pad.shape, X_test_pad.shape, config)
predictions_rnn = pd.read_csv("predictions_rnn.csv")

(75, 15) (25, 15) {'maxlen': 15, 'vocab_size': 225, 'embed_dim': 32}


In [2]:
# 4.1
X_train_t = torch.tensor(X_train_pad, dtype=torch.long)
X_test_t = torch.tensor(X_test_pad, dtype=torch.long)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

In [3]:
# 4.2
embed_dim = config["embed_dim"]

class SMSLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float32),
            freeze=False,
            padding_idx=0,
        )
        self.lstm = nn.LSTM(input_size=embed_dim, hidden_size=32, batch_first=True)
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        emb = self.embedding(x)
        out, (hidden, cell) = self.lstm(emb)
        logits = self.fc(hidden.squeeze(0))
        return torch.sigmoid(logits).squeeze(1)

torch.manual_seed(1)
model = SMSLSTM()

In [4]:
# 4.3
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(60):
    optimizer.zero_grad()
    y_proba_train = model(X_train_t)
    loss = loss_fn(y_proba_train, y_train_t)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}, loss={loss.item():.4f}")

Epoch 10, loss=0.5948
Epoch 20, loss=0.3402
Epoch 30, loss=0.0553
Epoch 40, loss=0.0092
Epoch 50, loss=0.0034
Epoch 60, loss=0.0021


In [5]:
# 4.4
model.eval()
with torch.no_grad():
    y_proba = model(X_test_t).numpy()
y_pred = (y_proba >= 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
recall = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
f1 = f1_score(y_test, y_pred, pos_label=1, zero_division=0)

print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-score: {f1:.2f}")

Accuracy: 1.00
Precision: 1.00
Recall: 1.00
F1-score: 1.00


In [6]:
# 4.5
results = pd.DataFrame({
    "text": test_text["clean_text"],
    "y_true": y_test,
    "y_pred": y_pred,
    "y_proba": y_proba,
})
results.to_csv("predictions_lstm.csv", index=False)
results.head()

,text,y_true,y_pred,y_proba
0,you have been selected to win a free voucher c...,1,1,0.998012
1,coffee tomorrow morning before work,0,0,0.002035
2,claim your free cash prize now urgent reply ne...,1,1,0.998063
3,win free cash now click link urgent claim requ...,1,1,0.998059
4,free cash prize waiting call now to claim urge...,1,1,0.998049


In [7]:
# 4.6
f1_rnn = f1_score(predictions_rnn["y_true"], predictions_rnn["y_pred"], pos_label=1, zero_division=0)
f1_lstm = f1_score(y_test, y_pred, pos_label=1, zero_division=0)

print(f"F1 RNN:  {f1_rnn:.2f}")
print(f"F1 LSTM: {f1_lstm:.2f}")

F1 RNN:  0.70
F1 LSTM: 1.00
